# H4rmony Qwen3-8B extreme-v2 and control evaluation

This is an **evaluation-only** Colab workflow. It locates the newest hash-verified `Qwen/Qwen3-8B` H4rmony R1 LoRA run already stored in Google Drive; it never starts SFT. It scores the unchanged base model and that saved adapter on all eight primary `extreme_v2` prompts and all six controls, using exact non-thinking `Yes`/`No` sequence scores. The four variable-cost controls use every configured value of $N$; the two zero-cost controls contain no `{cost}` field and are each scored once at $N=0$.

Each completed result bundle is first written under local `/content`, then copied beneath the source SFT run in Drive. Drive is flushed, remounted, and every artifact hash is checked through the fresh mount. Finally, both compact verified bundles are committed and pushed to GitHub. The adapter is always loaded on the immutable base-model revision recorded by the original SFT run.

Use an A100 40 GB (or larger). Before running, create a Colab secret named `GITHUB_TOKEN`, grant this notebook access to it, and give the token **Contents: read and write** permission for `shengweiming/value-misalignment`. The token is used only through a transient Git credential helper and is never printed or stored in the repository.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)
# Colab's optional torchao package can be incompatible with PEFT. This BF16
# evaluation does not use TorchAO quantization.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)
REPOSITORY_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("Repository commit:", REPOSITORY_COMMIT)

In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from google.colab import drive
drive.mount("/content/drive")

import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then retry."
assert torch.cuda.is_bf16_supported(), "This evaluation requires a BF16-capable GPU."
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / 2**30
assert gpu_memory_gib >= 38, "Select an A100 40 GB (or larger) runtime."
print(f"GPU: {gpu.name} ({gpu_memory_gib:.1f} GiB)")

The configuration below is the training signature used only to identify the existing adapter: Qwen3-8B, H4rmony R1-only data, rank-16 LoRA, three epochs, and seed 42. Changing an evaluation-only setting does not alter the adapter search. `FORCE_EVALUATION=False` independently reuses a primary or control result only when the source SFT completion hash, that suite's prompt hashes, the full $N$ grid, the evaluation protocol, and every result hash match.

In [ ]:
from google.colab import userdata
from scripts.harmony_sft import SFTConfig

DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/value-misalignment/harmony_r1_qwen3_8b")
FORCE_EVALUATION = False
PUBLISH_TO_GITHUB = True
GITHUB_REPOSITORY = "shengweiming/value-misalignment"
GITHUB_BRANCH = "main"

CONFIG = SFTConfig(
    output_root=Path("/content/evaluation-only-no-training"),
    require_google_drive=False,
    base_model="Qwen/Qwen3-8B",
    dataset_id="neovalle/H4rmony",
    max_length=1024,
    num_train_epochs=3,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.05,
    eval_batch_size=4,
    seed=42,
    cost_counts=(0, 1, 10, 100, 1_000, 10_000, 100_000, 1_000_000),
)

GITHUB_TOKEN = None
if PUBLISH_TO_GITHUB:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Add a Colab secret named GITHUB_TOKEN and grant this notebook access before evaluation."
        ) from exc
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "GITHUB_TOKEN is empty; GitHub publication is enabled and is required by this workflow."
        )
print("N values:", CONFIG.cost_counts)
CONFIG

In [ ]:
from IPython.display import Markdown, display
from scripts.harmony_sft import EXTREME_V2_TEMPLATES, build_extreme_v2_cases

primary_preview_cases = build_extreme_v2_cases(CONFIG.cost_counts)
assert len(primary_preview_cases) == len(EXTREME_V2_TEMPLATES) * len(CONFIG.cost_counts)
print("Primary templates:", EXTREME_V2_TEMPLATES)
print(f"Reviewing all {len(primary_preview_cases)} primary cases before inference.")
for case in primary_preview_cases:
    display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

In [ ]:
from scripts.harmony_sft import (
    EXTREME_V2_CONTROL_TEMPLATES,
    build_extreme_v2_control_cases,
)

control_preview_cases = build_extreme_v2_control_cases(CONFIG.cost_counts)
variable_control_count = 4 * len(CONFIG.cost_counts)
fixed_control_count = 2
assert len(control_preview_cases) == variable_control_count + fixed_control_count
fixed_controls = [
    case for case in control_preview_cases
    if "__zero_cost_ecological__" in case["template"]
]
assert len(fixed_controls) == 2
assert all(case["cost_count"] == 0 for case in fixed_controls)
print("Control templates:", EXTREME_V2_CONTROL_TEMPLATES)
print(
    f"Reviewing {len(control_preview_cases)} control cases: "
    f"{variable_control_count} variable-cost and {fixed_control_count} fixed zero-cost."
)
for case in control_preview_cases:
    display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

In [ ]:
from scripts.harmony_sft import run_extreme_v2_workflow

primary_workflow = run_extreme_v2_workflow(
    DRIVE_OUTPUT_ROOT,
    CONFIG,
    force_evaluation=FORCE_EVALUATION,
)
print("Verified SFT run:", primary_workflow.sft_artifacts.run_dir)
print("Final adapter:", primary_workflow.sft_artifacts.final_adapter_dir)
if primary_workflow.evaluation_reused:
    print("PRIMARY EVALUATION SKIPPED — reusing the matching hash-verified Drive bundle.")
else:
    print("PRIMARY EVALUATION COMPLETE — new results were verified after Drive remount.")
print("Verified primary Drive result:", primary_workflow.evaluation_artifacts.output_dir)
primary_workflow.validation

In [ ]:
import json
import pandas as pd
from IPython.display import Image

primary_eval_artifacts = primary_workflow.evaluation_artifacts
primary_metadata = json.loads(primary_eval_artifacts.metadata_path.read_text())
primary_scores = pd.read_csv(primary_eval_artifacts.raw_scores_path)
assert len(primary_scores) == 2 * len(primary_preview_cases)
assert set(primary_scores["model_role"]) == {"base", "aligned"}
primary_summary = primary_scores.pivot(
    index=["template", "cost_count"],
    columns="model_role",
    values=["p_implement", "semantic_logit_implement"],
).sort_index()
display(primary_summary)
display(pd.read_csv(primary_eval_artifacts.thresholds_path))
display(Image(filename=str(primary_eval_artifacts.plot_path)))
print("Primary rendered cases:", primary_eval_artifacts.rendered_cases_path)
print("Primary raw base + SFT scores:", primary_eval_artifacts.raw_scores_path)
print("Primary thresholds:", primary_eval_artifacts.thresholds_path)
print("Primary Drive completion manifest:", primary_eval_artifacts.complete_marker_path)

In [ ]:
from scripts.harmony_sft import run_extreme_v2_control_workflow

control_workflow = run_extreme_v2_control_workflow(
    DRIVE_OUTPUT_ROOT,
    CONFIG,
    force_evaluation=FORCE_EVALUATION,
)
if control_workflow.evaluation_reused:
    print("CONTROL EVALUATION SKIPPED — reusing the matching hash-verified Drive bundle.")
else:
    print("CONTROL EVALUATION COMPLETE — new results were verified after Drive remount.")
print("Verified control Drive result:", control_workflow.evaluation_artifacts.output_dir)
control_workflow.validation

In [ ]:
control_eval_artifacts = control_workflow.evaluation_artifacts
control_metadata = json.loads(control_eval_artifacts.metadata_path.read_text())
control_scores = pd.read_csv(control_eval_artifacts.raw_scores_path)
assert len(control_scores) == 2 * len(control_preview_cases)
assert set(control_scores["model_role"]) == {"base", "aligned"}
control_summary = control_scores.pivot(
    index=["template", "cost_count"],
    columns="model_role",
    values=["p_implement", "semantic_logit_implement"],
).sort_index()
display(control_summary)
display(pd.read_csv(control_eval_artifacts.thresholds_path))
display(Image(filename=str(control_eval_artifacts.plot_path)))
print("Control rendered cases:", control_eval_artifacts.rendered_cases_path)
print("Control raw base + SFT scores:", control_eval_artifacts.raw_scores_path)
print("Control thresholds:", control_eval_artifacts.thresholds_path)
print("Control Drive completion manifest:", control_eval_artifacts.complete_marker_path)

In [ ]:
from scripts.harmony_sft import publish_extreme_v2_results_to_github

primary_publication = None
control_publication = None
if PUBLISH_TO_GITHUB:
    primary_publication = publish_extreme_v2_results_to_github(
        primary_workflow.evaluation_artifacts,
        source_run_name=primary_workflow.sft_artifacts.run_dir.name,
        github_repository=GITHUB_REPOSITORY,
        branch=GITHUB_BRANCH,
        github_token=GITHUB_TOKEN,
        repo_root=REPO_DIR,
    )
    print("Primary GitHub publication verified:", primary_publication.html_url)
    print("Primary GitHub commit:", primary_publication.commit_sha)
    control_publication = publish_extreme_v2_results_to_github(
        control_workflow.evaluation_artifacts,
        source_run_name=control_workflow.sft_artifacts.run_dir.name,
        github_repository=GITHUB_REPOSITORY,
        branch=GITHUB_BRANCH,
        github_token=GITHUB_TOKEN,
        repo_root=REPO_DIR,
    )
    print("Control GitHub publication verified:", control_publication.html_url)
    print("Control GitHub commit:", control_publication.commit_sha)
else:
    print("GitHub publication disabled; both verified Drive bundles are intact.")

A successful run has three independent checks: the source SFT run matches the configured training signature and passes its original completion hashes; the primary bundle contains exactly one base and one aligned row for all 64 cases while the control bundle does so for all 34 cases, and each passes its own hashes after a fresh Drive remount; and the GitHub branch tip is read back after each push and must equal the local result commit. The four variable-cost controls contribute 32 cases, while the two fixed zero-cost controls contribute one $N=0$ case each. Existing matching Drive and GitHub bundles are reused only when their contents agree exactly. No step retrains the adapter, overwrites a prior result bundle, force-pushes, or places the GitHub token in repository state.